# Imports

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

ModuleNotFoundError: No module named 'pmdarima'

# Data

In [3]:
df_ts = pd.read_parquet('./data/pp-complete.parquet')

df_ts['year'] = df_ts['date'].dt.year
df_ts['log_price'] = np.log1p(df_ts['price'])
df_ts

NameError: name 'np' is not defined

# Functions

In [4]:
def plot_arima_forecast(series, category_name, n_periods=3, title_prefix=""):
    """
    Строит модель ARIMA, делает прогноз и визуализирует его.
    series: pd.Series - временной ряд средних price с годовым DateTimeIndex.
    category_name: str - имя категории для заголовков.
    n_periods: int - горизонт прогнозирования в годах.
    """
    if series.empty or len(series) < 10: # Нужно достаточно данных для ARIMA
        print(f"Недостаточно данных для {title_prefix} {category_name}. Пропуск.")
        return None, None

    plt.figure(figsize=(12, 6))
    plt.plot(series, label='Исторические данные (log_price)')
    plt.title(f'{title_prefix} Средняя log_price для {category_name}')
    plt.xlabel('Год')
    plt.ylabel('Средняя log_price')
    plt.show()

    # Проверка на стационарность
    result_adf = adfuller(series.dropna()) # Удаляем NaN если есть пропуски в годах
    print(f'\nADF Statistic for {category_name}: {result_adf[0]}')
    print(f'p-value: {result_adf[1]}')
    # Если p-value > 0.05, ряд, вероятно, нестационарен

    # Auto ARIMA
    # start_p, start_q, max_p, max_q можно настроить
    # D=1 может быть полезно, если тест ADF показывает нестационарность
    # m=1 для годовых данных (нет явной сезонности на этом уровне)
    try:
        model = pm.auto_arima(series.dropna(),
                              start_p=1, start_q=1,
                              test='adf',       # use adftest to find optimal 'd'
                              max_p=3, max_q=3, # maximum p and q
                              m=1,              # frequency of series (1 for annual data)
                              d=None,           # let model determine 'd'
                              seasonal=False,   # No seasonality for annual data / or let auto_arima decide by testing
                              start_P=0,
                              D=0,              # No seasonal differencing for annual data
                              trace=True,
                              error_action='ignore',
                              suppress_warnings=True,
                              stepwise=True)

        print(f"\nЛучшая модель ARIMA для {category_name}: {model.summary()}")

        # Прогноз
        forecast_log, conf_int_log = model.predict(n_periods=n_periods, return_conf_int=True)

        # Индексы для прогноза
        last_year = series.index.max().year
        forecast_index = pd.to_datetime([str(year) for year in range(last_year + 1, last_year + 1 + n_periods)])

        forecast_series_log = pd.Series(forecast_log, index=forecast_index)
        conf_int_df_log = pd.DataFrame(conf_int_log, index=forecast_index, columns=['lower_log_price', 'upper_log_price'])

        # Визуализация прогноза
        plt.figure(figsize=(12, 6))
        plt.plot(series, label='Исторические данные (log_price)')
        plt.plot(forecast_series_log, label='Прогноз (log_price)', color='red')
        plt.fill_between(forecast_index,
                         conf_int_df_log['lower_log_price'],
                         conf_int_df_log['upper_log_price'], color='pink', alpha=0.3, label='Доверительный интервал')
        plt.title(f'{title_prefix} Прогноз средней log_price для {category_name}')
        plt.xlabel('Год')
        plt.ylabel('Средняя log_price')
        plt.legend()
        plt.show()

        # Обратное преобразование в цены
        forecast_price = np.expm1(forecast_series_log)
        historical_price = np.expm1(series)
        
        print(f"\nПрогноз цен для {category_name} на {n_periods} года:")
        print(forecast_price)
        
        return model, forecast_price

    except Exception as e:
        print(f"Ошибка при построении модели для {category_name}: {e}")
        return None, None

# Forecast

## Общая средняя цена

In [5]:
df_agg_total = df_ts.groupby('year')['log_price'].mean()
# Преобразуем индекс в DateTimeIndex (годовой)
df_agg_total.index = pd.to_datetime(df_agg_total.index, format='%Y')
df_agg_total = df_agg_total.asfreq('AS') # 'AS' - Annual, start of year

model_total, forecast_total = plot_arima_forecast(df_agg_total, "Всех типов недвижимости")

KeyError: 'Column not found: log_price'